<a href="https://colab.research.google.com/github/zach8421/portable_waste_sorting/blob/main/portable_waste_sorting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Portable Waste-Sorting Information for Seattle Residents

**IMT 542 — I3 Assignment**

This notebook takes the waste-disposal guidance Seattle Public Utilities publishes through its [Where Does It Go?](https://www.seattle.gov/utilities/your-services/collection-and-disposal/where-does-it-go) tool and reframes it as a *portable* JSON structure that can answer item-level questions ("does this go in recycling, compost, or garbage?") from any interface.

The JSON schema follows the G3 proposal: each item records its name, synonyms, disposal category, preparation steps, explanation, and a link back to the authoritative source. Applying this structure to real items lets us build a searchable tree, a summary chart, and a simple lookup function on top of the data.

The notebook loads a seed dataset (bundled as `data/items.json`) and visualizes it. The final section includes an optional scraper that fetches additional items from the live site using a headless browser — the Seattle page is a JavaScript single-page app, so plain `requests` won't work there. The seed file is enough for everything above the scraper cell to run end-to-end.


In [ ]:
# Install pandas and plotly. Colab has both pre-installed, but this makes the notebook portable.
!pip install -q pandas plotly requests


In [ ]:
# Imports. pandas handles the tabular transformations, plotly draws the tree, requests pulls the JSON.
import json
import requests
import pandas as pd
import plotly.graph_objects as go


## 1. Load the portable information structure

`items.json` holds two arrays: `categories` (the tree skeleton) and `items` (each item as one record in the G3 schema). We try loading it from the GitHub raw URL first so the notebook runs in Colab with no setup, and fall back to the local file when working on the repo directly.


In [ ]:
# Load items.json. Update RAW_URL to point at your own fork after pushing.
RAW_URL = "https://raw.githubusercontent.com/zach8421/portable_waste_sorting/main/data/items.json"
LOCAL_PATH = "data/items.json"

try:
    data = requests.get(RAW_URL, timeout=10).json()
    print(f"Loaded from GitHub: {len(data['items'])} items, {len(data['categories'])} categories")
except Exception:
    with open(LOCAL_PATH) as f:
        data = json.load(f)
    print(f"Loaded from local file: {len(data['items'])} items, {len(data['categories'])} categories")

print("Source:", data['metadata']['source'])
print("Last updated:", data['metadata']['last_updated'])


## 2. Flatten items into a DataFrame

`pd.json_normalize` handles the conversion from the nested JSON records to a flat table. This is the same pattern used in the I3 example notebook's SEC EDGAR cell.


In [ ]:
# Flatten the items list into a DataFrame, one row per item.
items_df = pd.json_normalize(data['items'])

# synonyms is a list-valued column — join to a string for display purposes.
items_df['synonyms_str'] = items_df['synonyms'].apply(lambda xs: ", ".join(xs))

print("Shape:", items_df.shape)
items_df[['item_name', 'disposal_category', 'material', 'synonyms_str']].head(10)


In [ ]:
# How many items land in each disposal category? This tells us how the data is distributed.
items_df['disposal_category'].value_counts()


## 3. Visualize the category tree

A sunburst chart is a natural fit for a portable information structure like this: top-level categories in the inner ring, sub-categories and items radiating outward. Hovering shows the item name and its disposal category.


In [ ]:
# Build the sunburst nodes. Every category and every item becomes a node;
# each item's parent is the last element of its category_path.
labels, parents, ids, hover = [], [], [], []

for c in data['categories']:
    ids.append(c['category_id'])
    labels.append(c['category_name'])
    parents.append(c['parent'] if c['parent'] else "")
    hover.append(f"Category: {c['category_name']}")

# Colour each leaf by disposal category to make the output legible at a glance.
category_colour = {
    'recycle': '#2b8a3e',
    'compost': '#8a6d1a',
    'garbage': '#495057',
    'hazardous_waste': '#c92a2a',
    'transfer_station': '#1864ab',
    'special_pickup': '#6741d9',
    'donate': '#0c8599',
}
colours = ['#dee2e6'] * len(data['categories'])  # neutral grey for category rings

for item in data['items']:
    ids.append(item['item_id'])
    labels.append(item['item_name'])
    parents.append(item['category_path'][-1])
    hover.append(
        f"<b>{item['item_name']}</b><br>"
        f"Disposal: {item['disposal_category']}<br>"
        f"Material: {item['material']}<br>"
        f"{item['explanation']}"
    )
    colours.append(category_colour.get(item['disposal_category'], '#868e96'))

fig = go.Figure(go.Sunburst(
    labels=labels,
    parents=parents,
    ids=ids,
    hovertext=hover,
    hoverinfo='text',
    marker=dict(colors=colours),
    branchvalues='total',
))
fig.update_layout(
    title="Seattle Waste Disposal — Category Tree",
    width=800, height=700,
    margin=dict(t=60, l=10, r=10, b=10),
)
fig.show()


## 4. How does the data split by disposal stream?

A quick bar chart of items grouped by `disposal_category` shows which streams dominate the current dataset. As the scraper fills in more items, this distribution will shift.


In [ ]:
counts = items_df['disposal_category'].value_counts().reset_index()
counts.columns = ['disposal_category', 'count']

bar = go.Figure(go.Bar(
    x=counts['disposal_category'],
    y=counts['count'],
    marker_color=[category_colour.get(c, '#868e96') for c in counts['disposal_category']],
    text=counts['count'],
    textposition='outside',
))
bar.update_layout(
    title="Items per disposal category",
    xaxis_title="Disposal category",
    yaxis_title="Number of items",
    width=800, height=450,
    plot_bgcolor='#fafafa',
)
bar.show()


## 5. Item lookup — the "portable" use case

The whole point of the G3 schema is that the same JSON can power any interface. Here's a minimal Python function that answers "where does this go?" using item names or synonyms. Swap the body for JavaScript, Swift, or an API endpoint and the structure still holds.


In [ ]:
def lookup(query, data=data):
    """Find matching items by name or synonym and return disposal guidance."""
    q = query.lower().strip()
    hits = []
    for item in data['items']:
        haystack = [item['item_name'].lower()] + [s.lower() for s in item['synonyms']]
        if any(q in h or h in q for h in haystack):
            hits.append(item)
    return hits

def print_guidance(query):
    results = lookup(query)
    if not results:
        print(f"No match for '{query}'. Try a different term or check the source URL.")
        return
    for r in results:
        print(f"\n→ {r['item_name'].upper()}")
        print(f"   Disposal: {r['disposal_category']}")
        print(f"   How:      {r['disposal_conditions']}")
        print(f"   Prep:     {r['preparation_required']}")
        print(f"   Why:      {r['explanation']}")
        print(f"   Source:   {r['source_reference']}")

print_guidance("batteries")
print_guidance("pizza box")
print_guidance("grocery bag")


## 6. (Optional) Expand the dataset with a headless-browser scraper

The seed dataset above is enough to demonstrate the schema and all of the analyses. To cover every item on the live SPU page, the cell below uses **Playwright** to render the JavaScript SPA and pull item detail pages one at a time. This cell runs in Colab where Playwright's Chromium is not blocked; local runs may need tweaking.

Running it is not required for the assignment — it's how the seed data would be expanded in practice. Set `RUN_SCRAPER = True` below to enable it.


In [ ]:
RUN_SCRAPER = False  # set True in Colab to expand the dataset

if RUN_SCRAPER:
    !pip install -q playwright
    !playwright install chromium 2>&1 | tail -2

    from playwright.sync_api import sync_playwright
    import time

    BASE = "https://www.seattle.gov/utilities/your-services/collection-and-disposal/where-does-it-go"

    scraped = []
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(BASE + "#/categories", wait_until='networkidle', timeout=30000)

        # Pull every item link from the A-to-Z / categories listing.
        item_links = page.eval_on_selector_all(
            "a[href*='#/item/']",
            "els => els.map(e => ({ name: e.textContent.trim(), href: e.href }))"
        )
        # De-duplicate by href.
        seen, unique = set(), []
        for link in item_links:
            if link['href'] not in seen:
                seen.add(link['href']); unique.append(link)
        print(f"Found {len(unique)} item links.")

        # Visit each and capture the main disposal panel text.
        for i, link in enumerate(unique):
            try:
                page.goto(link['href'], wait_until='networkidle', timeout=20000)
                body = page.locator("main").inner_text()
                scraped.append({
                    'item_name': link['name'],
                    'source_reference': link['href'],
                    'raw_text': body,
                })
                if (i+1) % 10 == 0: print(f"  scraped {i+1}/{len(unique)}")
                time.sleep(0.3)  # be polite
            except Exception as e:
                print(f"  skipped {link['name']}: {e}")
        browser.close()

    # scraped is a list of {item_name, source_reference, raw_text}.
    # The next step would parse each raw_text block into the full G3 schema
    # (disposal_category, preparation_required, etc.) using keyword rules or an LLM.
    with open('data/items_raw_scrape.json', 'w') as f:
        json.dump(scraped, f, indent=2)
    print(f"Saved {len(scraped)} scraped items to data/items_raw_scrape.json")
else:
    print("Scraper disabled. Set RUN_SCRAPER = True to run.")


## Summary

- The G3 schema serialised cleanly to JSON and loaded into a DataFrame in two lines.
- A sunburst chart makes the category → item tree immediately scannable.
- A 15-line `lookup()` function provides the portable "where does this go?" use case.
- Expanding to every SPU item is a scraper away, and the schema is the same whether there are 20 items or 2,000.
